In [61]:
import pandas as pd

In [62]:
price_dataset = pd.read_csv("../data/raw/DE/Germany.csv")
price_dataset["datetime"] = pd.to_datetime(price_dataset["Datetime (UTC)"])

In [63]:
price_dataset.head()

,Country,ISO3 Code,Datetime (UTC),Datetime (Local),Price (EUR/MWhe),datetime
0,Germany,DEU,2015-01-01 00:00:00,2015-01-01 01:00:00,22.34,2015-01-01 00:00:00
1,Germany,DEU,2015-01-01 01:00:00,2015-01-01 02:00:00,22.34,2015-01-01 01:00:00
2,Germany,DEU,2015-01-01 02:00:00,2015-01-01 03:00:00,22.34,2015-01-01 02:00:00
3,Germany,DEU,2015-01-01 03:00:00,2015-01-01 04:00:00,22.34,2015-01-01 03:00:00
4,Germany,DEU,2015-01-01 04:00:00,2015-01-01 05:00:00,22.34,2015-01-01 04:00:00


In [64]:
price_dataset = price_dataset.drop(columns=["Country", "ISO3 Code", 'Datetime (Local)', 'Datetime (UTC)'])

In [65]:
price_dataset.head()

,Price (EUR/MWhe),datetime
0,22.34,2015-01-01 00:00:00
1,22.34,2015-01-01 01:00:00
2,22.34,2015-01-01 02:00:00
3,22.34,2015-01-01 03:00:00
4,22.34,2015-01-01 04:00:00


In [66]:
solar_dataset = pd.read_csv("../data/raw/DE/solar-raw.csv")

In [67]:
solar_dataset.head()

,Unnamed: 0,solar_generation_MW
0,2022-01-01 00:00:00+00:00,1.745
1,2022-01-01 01:00:00+00:00,1.770
2,2022-01-01 02:00:00+00:00,1.795
3,2022-01-01 03:00:00+00:00,1.770
4,2022-01-01 04:00:00+00:00,1.820


In [68]:
solar_dataset.columns = ["datetime", "solar_generation_MW"]

In [69]:
solar_dataset.head()

,datetime,solar_generation_MW
0,2022-01-01 00:00:00+00:00,1.745
1,2022-01-01 01:00:00+00:00,1.770
2,2022-01-01 02:00:00+00:00,1.795
3,2022-01-01 03:00:00+00:00,1.770
4,2022-01-01 04:00:00+00:00,1.820


In [70]:
solar_dataset["datetime"] = pd.to_datetime(solar_dataset["datetime"], utc=True).dt.tz_localize(None)

In [71]:
solar_dataset.isna().sum()

datetime               0
solar_generation_MW    0
dtype: int64

In [72]:
meteo_dataset = pd.read_csv("../data/raw/DE/meteo-raw-germany.csv")

In [73]:
meteo_dataset.head()

,time,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,2022-01-01T00:00,10.3,13.5,241,100,0.0,0.1
1,2022-01-01T01:00,10.9,13.2,248,99,0.0,0.1
2,2022-01-01T02:00,10.2,13.7,247,100,0.0,0.0
3,2022-01-01T03:00,9.9,14.9,238,100,0.0,0.0
4,2022-01-01T04:00,9.2,12.8,230,100,0.0,0.0


In [74]:
meteo_dataset["datetime"] = pd.to_datetime(meteo_dataset["time"])

In [75]:
meteo_dataset.columns

Index(['time', 'temperature_2m', 'wind_speed_10m', 'wind_direction_10m',
       'cloud_cover', 'shortwave_radiation', 'precipitation', 'datetime'],
      dtype='object')

In [76]:
meteo_dataset = meteo_dataset.drop(columns=["time"])

In [77]:
meteo_dataset.head()

,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation,datetime
0,10.3,13.5,241,100,0.0,0.1,2022-01-01 00:00:00
1,10.9,13.2,248,99,0.0,0.1,2022-01-01 01:00:00
2,10.2,13.7,247,100,0.0,0.0,2022-01-01 02:00:00
3,9.9,14.9,238,100,0.0,0.0,2022-01-01 03:00:00
4,9.2,12.8,230,100,0.0,0.0,2022-01-01 04:00:00


In [78]:
merged = price_dataset.merge(solar_dataset, on="datetime", how="inner")
merged = merged.merge(meteo_dataset, on="datetime", how="inner")
merged = merged.sort_values("datetime").reset_index(drop=True)

print(merged.shape)
merged.head()

(35064, 9)


,Price (EUR/MWhe),datetime,solar_generation_MW,temperature_2m,wind_speed_10m,wind_direction_10m,cloud_cover,shortwave_radiation,precipitation
0,31.39,2022-01-01 00:00:00,1.745,10.3,13.5,241,100,0.0,0.1
1,26.88,2022-01-01 01:00:00,1.770,10.9,13.2,248,99,0.0,0.1
2,27.36,2022-01-01 02:00:00,1.795,10.2,13.7,247,100,0.0,0.0
3,20.64,2022-01-01 03:00:00,1.770,9.9,14.9,238,100,0.0,0.0
4,27.80,2022-01-01 04:00:00,1.820,9.2,12.8,230,100,0.0,0.0


In [80]:
merged.to_csv("../data/processed/germany_merged.csv", index=False)